# API Tools

For this section we'll just get a default ReAct agent to run our tools; this will let us focus our debugging in the tools themselves rather than the agent.

In [28]:
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [9]:
!uv pip install python-dotenv requests

from dotenv import load_dotenv, find_dotenv
import requests, os
from urllib.parse import urljoin
from typing import Literal

# Load environment variables from .env file
load_dotenv(find_dotenv())
FOOTBALL_API_KEY = os.getenv('FOOTBALL_API_KEY')
FOOTBALL_API_BASE_URL = "https://v3.football.api-sports.io/"
FOOTBALL_API_HOST = "v3.football.api-sports.io"

def call_football_api(method: Literal["GET", "OPTIONS", "HEAD", "POST", "PUT", "PATCH", "DELETE"], endpoint: str, data={}, params=None):
  """
  Call the Football API with the given endpoint and parameters.
  """
  url = urljoin(FOOTBALL_API_BASE_URL, endpoint)
  headers = {
    'x-rapidapi-key': FOOTBALL_API_KEY,
    'x-rapidapi-host': FOOTBALL_API_HOST
  }
  response = requests.request(method, url, headers=headers, data=data, params=params)
  return response.json()

response = call_football_api("GET", "status")

print(response)

Audited 2 packages in 4ms
{'get': 'status', 'parameters': [], 'errors': [], 'results': 0, 'paging': {'current': 1, 'total': 1}, 'response': {'account': {'firstname': 'Vasco', 'lastname': 'Peleteiro', 'email': 'vasco@augustalabs.ai'}, 'subscription': {'plan': 'Pro', 'end': '2025-07-05T20:58:08+00:00', 'active': True}, 'requests': {'current': 0, 'limit_day': 7500}}}


We need to implement a tool that can query the [Football API](https://www.api-football.com/documentation-v3) to get information about football leagues, teams, players, and matches.

1. Classificações de equipas — <https://www.api-football.com/documentation-v3#tag/Standings/operation/get-standings>
2. Próximos jogos — <https://www.api-football.com/documentation-v3#tag/Fixtures/operation/get-fixtures> see `next` parameter
3. Últimos jogos — <https://www.api-football.com/documentation-v3#tag/Fixtures/operation/get-fixtures> see `last` parameter
4. Jogos específicos (ex: SLB vs SCP para a liga em 2012/13) — we need to get the fixture ID first, then use it to get the match details.
5. Resultados de jogos específicos
6. Eventos de jogos específicos (ex.: golos, cartões, substituições).
7. Estatísticas de jogadores (ex: número de golos do jogador X na época Y)

Qualquer equipa, jogo ou jogador das top 7 ligas europeias + das 3 competições europeias.

Extra:

8. Odds
9. H2H
10. ...


## API Football

There doesn't seem to be a OpenAPI spec for this API, but we'll use the documentation to implement the tools we need and see how it goes.

### 0. Setup

To start we'll need to get the IDs for all leagues and teams for the top 7 leagues in Europe, as well as the European competitions. They are:
- Premier League (England)
- La Liga (Spain)
- Serie A (Italy)
- Bundesliga (Germany)
- Ligue 1 (France)
- Primeira Liga (Portugal)
- Eredivisie (Netherlands)
- UEFA Champions League
- UEFA Europa League
- UEFA Conference League

We should get the cups as well:
- FA Cup (England)
- Copa del Rey (Spain)
- Coppa Italia (Italy)
- DFB Pokal (Germany)
- Coupe de France (France)
- Taça de Portugal (Portugal)
- KNVB Beker (Netherlands)
- UEFA Super Cup

We'll save these in a JSON dictionary to save on API calls later and define local functions to retrieve them.

In [10]:
leagues_en = call_football_api("GET", "leagues", params={"search": "England"})

In [12]:
leagues_en

{'get': 'leagues',
 'parameters': {'search': 'England'},
 'errors': [],
 'results': 46,
 'paging': {'current': 1, 'total': 1},
 'response': [{'league': {'id': 39,
    'name': 'Premier League',
    'type': 'League',
    'logo': 'https://media.api-sports.io/football/leagues/39.png'},
   'country': {'name': 'England',
    'code': 'GB-ENG',
    'flag': 'https://media.api-sports.io/flags/gb-eng.svg'},
   'seasons': [{'year': 2010,
     'start': '2010-08-14',
     'end': '2011-05-17',
     'current': False,
     'coverage': {'fixtures': {'events': True,
       'lineups': True,
       'statistics_fixtures': False,
       'statistics_players': False},
      'standings': True,
      'players': True,
      'top_scorers': True,
      'top_assists': True,
      'top_cards': True,
      'injuries': False,
      'predictions': True,
      'odds': False}},
    {'year': 2011,
     'start': '2011-08-13',
     'end': '2012-05-13',
     'current': False,
     'coverage': {'fixtures': {'events': True,
   

In [14]:
leagues = call_football_api("GET", "leagues", params={})
leagues

{'get': 'leagues',
 'parameters': [],
 'errors': [],
 'results': 1186,
 'paging': {'current': 1, 'total': 1},
 'response': [{'league': {'id': 4,
    'name': 'Euro Championship',
    'type': 'Cup',
    'logo': 'https://media.api-sports.io/football/leagues/4.png'},
   'country': {'name': 'World', 'code': None, 'flag': None},
   'seasons': [{'year': 2008,
     'start': '2008-06-07',
     'end': '2008-06-29',
     'current': False,
     'coverage': {'fixtures': {'events': True,
       'lineups': True,
       'statistics_fixtures': False,
       'statistics_players': False},
      'standings': False,
      'players': True,
      'top_scorers': True,
      'top_assists': True,
      'top_cards': True,
      'injuries': False,
      'predictions': True,
      'odds': False}},
    {'year': 2012,
     'start': '2012-06-08',
     'end': '2012-07-01',
     'current': False,
     'coverage': {'fixtures': {'events': True,
       'lineups': True,
       'statistics_fixtures': False,
       'statisti

In [16]:
def extract_league_info(leagues_response):
    """
    Extract league name, id, and country from the Football API leagues response.
    
    Args:
        leagues_response: JSON response from the Football API leagues endpoint
        
    Returns:
        List of dictionaries containing league info
    """
    league_info = []
    
    if 'response' in leagues_response:
        for item in leagues_response['response']:
            league = item.get('league', {})
            country = item.get('country', {})
            
            league_data = {
                'id': league.get('id'),
                'name': league.get('name'),
                'country': country.get('name')
            }
            league_info.append(league_data)
    
    return league_info

# Test
leagues_parsed = extract_league_info(leagues)
for league in leagues_parsed:
    print(f"ID: {league['id']}, Name: {league['name']}, Country: {league['country']}")

ID: 4, Name: Euro Championship, Country: World
ID: 21, Name: Confederations Cup, Country: World
ID: 61, Name: Ligue 1, Country: France
ID: 144, Name: Jupiler Pro League, Country: Belgium
ID: 71, Name: Serie A, Country: Brazil
ID: 39, Name: Premier League, Country: England
ID: 78, Name: Bundesliga, Country: Germany
ID: 135, Name: Serie A, Country: Italy
ID: 88, Name: Eredivisie, Country: Netherlands
ID: 94, Name: Primeira Liga, Country: Portugal
ID: 140, Name: La Liga, Country: Spain
ID: 179, Name: Premiership, Country: Scotland
ID: 180, Name: Championship, Country: Scotland
ID: 1, Name: World Cup, Country: World
ID: 803, Name: Asian Games, Country: World
ID: 804, Name: Caribbean Cup, Country: World
ID: 62, Name: Ligue 2, Country: France
ID: 2, Name: UEFA Champions League, Country: World
ID: 311, Name: 1st Division, Country: Albania
ID: 310, Name: Superliga, Country: Albania
ID: 186, Name: Ligue 1, Country: Algeria
ID: 187, Name: Ligue 2, Country: Algeria
ID: 42, Name: League Two, Count

Let's extract the leagues we want:

In [19]:
# Define the leagues we want with their countries
target_leagues = {
  "Premier League": "England",
  "La Liga": "Spain", 
  "Serie A": "Italy",
  "Bundesliga": "Germany",
  "Ligue 1": "France",
  "Primeira Liga": "Portugal",
  "Eredivisie": "Netherlands",
  "UEFA Champions League": "World",
  "UEFA Europa League": "World",
  "UEFA Europa Conference League": "World",
  "FA Cup": "England",
  "Copa del Rey": "Spain",
  "Coppa Italia": "Italy",
  "DFB Pokal": "Germany",
  "Coupe de France": "France",
  "Taça de Portugal": "Portugal",
  "KNVB Beker": "Netherlands",
  "UEFA Super Cup": "World"
}

selected_leagues_filtered = []

for league in leagues_parsed:
  league_name = league['name']
  league_country = league['country']
  
  if league_name in target_leagues and target_leagues[league_name] == league_country:
    selected_leagues_filtered.append(league)
    print(f"Selected League: {league_name} (ID: {league['id']}, Country: {league_country})")

print(f"\nTotal selected leagues: {len(selected_leagues_filtered)}")

Selected League: Ligue 1 (ID: 61, Country: France)
Selected League: Premier League (ID: 39, Country: England)
Selected League: Bundesliga (ID: 78, Country: Germany)
Selected League: Serie A (ID: 135, Country: Italy)
Selected League: Eredivisie (ID: 88, Country: Netherlands)
Selected League: Primeira Liga (ID: 94, Country: Portugal)
Selected League: La Liga (ID: 140, Country: Spain)
Selected League: UEFA Champions League (ID: 2, Country: World)
Selected League: Coupe de France (ID: 66, Country: France)
Selected League: FA Cup (ID: 45, Country: England)
Selected League: DFB Pokal (ID: 81, Country: Germany)
Selected League: UEFA Europa League (ID: 3, Country: World)
Selected League: UEFA Super Cup (ID: 531, Country: World)
Selected League: Taça de Portugal (ID: 96, Country: Portugal)
Selected League: Coppa Italia (ID: 137, Country: Italy)
Selected League: KNVB Beker (ID: 90, Country: Netherlands)
Selected League: Copa del Rey (ID: 143, Country: Spain)
Selected League: UEFA Europa Conferen

In [20]:
# Save the selected leagues to a dictionary mapping the name to the ID
selected_leagues_dict = {league['name']: league['id'] for league in selected_leagues_filtered}
print(selected_leagues_dict)
# Save the selected leagues to a JSON file
import json
with open('selected_leagues.json', 'w') as f:
    json.dump(selected_leagues_dict, f, indent=4)
print("Selected leagues saved to 'selected_leagues.json'")

{'Ligue 1': 61, 'Premier League': 39, 'Bundesliga': 78, 'Serie A': 135, 'Eredivisie': 88, 'Primeira Liga': 94, 'La Liga': 140, 'UEFA Champions League': 2, 'Coupe de France': 66, 'FA Cup': 45, 'DFB Pokal': 81, 'UEFA Europa League': 3, 'UEFA Super Cup': 531, 'Taça de Portugal': 96, 'Coppa Italia': 137, 'KNVB Beker': 90, 'Copa del Rey': 143, 'UEFA Europa Conference League': 848}
Selected leagues saved to 'selected_leagues.json'


In [30]:
# Define function to get league ID by name
def get_league_id_by_name(league_name: str) -> int | None:
    """
    Get the league ID by its name.
    
    Args:
        league_name: Name of the league
        
    Returns:
        League ID if found, otherwise None
    """
    leagues = json.load(open('selected_leagues.json'))
    for league in leagues:
        if league['name'] == league_name:
            return league['id']
    
    # If the league is not found, partial match the name
    for league in leagues:
        if league_name.lower() in league['name'].lower():
            return league['id']
    
    # If no match is found, fetch it from the API:
    logger.warning(f"League '{league_name}' not found in selected leagues. Fetching from API...")
    response = call_football_api("GET", "leagues", params={"search": league_name})
    if 'response' in response and len(response['response']) > 0:
        leagues = extract_league_info(response)
        for league in leagues:
            if league['name'].lower() == league_name.lower():
                return league['id']

    return None

def list_leagues() -> list[str]:
    """
    List all available leagues.
    
    Returns:
        List of league names
    """
    leagues = json.load(open('selected_leagues.json'))
    return [league['name'] for league in leagues]

### 1. Standings